In [ ]:
!pip install mediapipe -q

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

from tensorflow.keras.layers import (
    Dense,
    Dropout,
    GlobalAveragePooling2D
)

from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau,
    ModelCheckpoint
)

from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix

print("TensorFlow Version:", tf.__version__)
print("GPU Available:", tf.config.list_physical_devices('GPU'))

TensorFlow Version: 2.20.0
GPU Available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [ ]:
from tensorflow.keras import mixed_precision

mixed_precision.set_global_policy('mixed_float16')

print("Mixed Precision Enabled")

Mixed Precision Enabled


In [ ]:
import os

os.environ["KAGGLE_USERNAME"] = "matsunichi"
os.environ["KAGGLE_KEY"] = "KGAT_72f3e2fa631852fa7849713e3246daab"

In [ ]:
!pip install -q kaggle

In [ ]:
!kaggle datasets download -d akashshingha850/mrl-eye-dataset

Dataset URL: https://www.kaggle.com/datasets/akashshingha850/mrl-eye-dataset
License(s): MIT
mrl-eye-dataset.zip: Skipping, found more recently modified local copy (use --force to force download)


In [ ]:
!unzip -oq /content/mrl-eye-dataset.zip -d /content/

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

for root, dirs, files in os.walk("/content"):
    print(root)

/content
/content/.config
/content/.config/configurations
/content/.config/logs
/content/.config/logs/2026.06.04
/content/data
/content/data/val
/content/data/val/awake
/content/data/val/sleepy
/content/data/test
/content/data/test/awake
/content/data/test/sleepy
/content/data/train
/content/data/train/awake
/content/data/train/sleepy
/content/drive
/content/drive/.shortcut-targets-by-id
/content/drive/MyDrive
/content/drive/MyDrive/konser feast hindia
/content/drive/MyDrive/roblok
/content/drive/MyDrive/roblok 2
/content/drive/MyDrive/tugas meymey
/content/drive/MyDrive/video balai magang
/content/drive/MyDrive/dokumen bu fitri
/content/drive/MyDrive/balai
/content/drive/MyDrive/Google Earth
/content/drive/MyDrive/Bahan Cyber Security
/content/drive/MyDrive/Colab Notebooks
/content/drive/.Trash-0
/content/drive/.Trash-0/files
/content/drive/.Trash-0/info
/content/drive/.Encrypted
/content/drive/.Encrypted/.shortcut-targets-by-id
/content/drive/.Encrypted/MyDrive
/content/drive/.Encryp

In [ ]:
import os

BASE_DIR = "/content/data"

TRAIN_DIR = os.path.join(BASE_DIR, "train")
VAL_DIR   = os.path.join(BASE_DIR, "val")
TEST_DIR  = os.path.join(BASE_DIR, "test")

print(TRAIN_DIR)
print(VAL_DIR)
print(TEST_DIR)

/content/data/train
/content/data/val
/content/data/test


In [ ]:
import os

awake_count = len(os.listdir(os.path.join(TRAIN_DIR, "awake")))
sleepy_count = len(os.listdir(os.path.join(TRAIN_DIR, "sleepy")))

print("Awake:", awake_count)
print("Sleepy:", sleepy_count)

Awake: 25770
Sleepy: 25167


In [ ]:
IMG_SIZE = 224
BATCH_SIZE = 128

INITIAL_EPOCHS = 5
FINE_TUNE_EPOCHS = 10

NUM_CLASSES = 2

In [ ]:
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=10,
    zoom_range=0.15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    brightness_range=[0.8,1.2],
    horizontal_flip=True
)

val_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input
)

test_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input
)

In [ ]:
train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='binary',
    shuffle=True
)

val_generator = val_datagen.flow_from_directory(
    VAL_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='binary',
    shuffle=False
)

test_generator = test_datagen.flow_from_directory(
    TEST_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='binary',
    shuffle=False
)

print(train_generator.class_indices)

Found 50937 images belonging to 2 classes.
Found 16980 images belonging to 2 classes.
Found 16981 images belonging to 2 classes.
{'awake': 0, 'sleepy': 1}


In [ ]:
base_model = MobileNetV2(
    input_shape=(224,224,3),
    include_top=False,
    weights='imagenet'
)

base_model.trainable = False

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [ ]:
x = base_model.output

x = GlobalAveragePooling2D()(x)

x = Dense(
    256,
    activation='relu'
)(x)

x = Dropout(0.3)(x)

output = Dense(
    1,
    activation='sigmoid'
)(x)

model = Model(
    inputs=base_model.input,
    outputs=output
)

model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1 (Conv2D)      │ (None, 112, 112,  │        864 │ input_layer[0][0] │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bn_Conv1            │ (None, 112, 112,  │        128 │ Conv1[0][0]       │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1_relu (ReLU)   │ (None, 112, 112,  │          0 │ bn_Conv1[0][0]    │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        288 │ Conv1_relu[0][0]  │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        128 │ expanded_conv_de… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │          0 │ expanded_conv_de… │
│ (ReLU)              │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │        512 │ expanded_conv_de… │
│ (Conv2D)            │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │         64 │ expanded_conv_pr… │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand      │ (None, 112, 112,  │      1,536 │ expanded_conv_pr… │
│ (Conv2D)            │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_BN   │ (None, 112, 112,  │        384 │ block_1_expand[0… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_relu │ (None, 112, 112,  │          0 │ block_1_expand_B… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_pad         │ (None, 113, 113,  │          0 │ block_1_expand_r… │
│ (ZeroPadding2D)     │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise   │ (None, 56, 56,    │        864 │ block_1_pad[0][0] │
│ (DepthwiseConv2D)   │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │        384 │ block_1_depthwis… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │          0 │ block_1_depthwis… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_project     │ (None, 56, 56,    │      2,304 │ block_1_depthwis

 Total params: 2,586,177 (9.87 MB)

 Trainable params: 328,193 (1.25 MB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [ ]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [ ]:
checkpoint = ModelCheckpoint(
    "mobilenetv2_eye_best.h5",
    monitor='val_accuracy',
    save_best_only=True,
    verbose=1
)

earlystop = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.2,
    patience=2,
    verbose=1
)

In [ ]:
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=INITIAL_EPOCHS,
    callbacks=[
        checkpoint,
        earlystop,
        reduce_lr
    ]
)

Epoch 1/5
398/398 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.9221 - loss: 0.1930
Epoch 1: val_accuracy improved from None to 0.96131, saving model to mobilenetv2_eye_best.h5



Epoch 1: finished saving model to mobilenetv2_eye_best.h5
398/398 ━━━━━━━━━━━━━━━━━━━━ 1023s 2s/step - accuracy: 0.9479 - loss: 0.1407 - val_accuracy: 0.9613 - val_loss: 0.1108 - learning_rate: 0.0010
Epoch 2/5
398/398 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.9618 - loss: 0.1039
Epoch 2: val_accuracy improved from 0.96131 to 0.96720, saving model to mobilenetv2_eye_best.h5



Epoch 2: finished saving model to mobilenetv2_eye_best.h5
398/398 ━━━━━━━━━━━━━━━━━━━━ 763s 2s/step - accuracy: 0.9626 - loss: 0.1028 - val_accuracy: 0.9672 - val_loss: 0.0931 - learning_rate: 0.0010
Epoch 3/5
398/398 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.9665 - loss: 0.0938
Epoch 3: val_accuracy improved from 0.96720 to 0.96908, saving model to mobilenetv2_eye_best.h5



Epoch 3: finished saving model to mobilenetv2_eye_best.h5
398/398 ━━━━━━━━━━━━━━━━━━━━ 712s 2s/step - accuracy: 0.9655 - loss: 0.0951 - val_accuracy: 0.9691 - val_loss: 0.0866 - learning_rate: 0.0010
Epoch 4/5
398/398 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.9679 - loss: 0.0867
Epoch 4: val_accuracy improved from 0.96908 to 0.96914, saving model to mobilenetv2_eye_best.h5



Epoch 4: finished saving model to mobilenetv2_eye_best.h5
398/398 ━━━━━━━━━━━━━━━━━━━━ 736s 2s/step - accuracy: 0.9664 - loss: 0.0897 - val_accuracy: 0.9691 - val_loss: 0.0834 - learning_rate: 0.0010
Epoch 5/5
398/398 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.9693 - loss: 0.0826
Epoch 5: val_accuracy improved from 0.96914 to 0.97079, saving model to mobilenetv2_eye_best.h5



Epoch 5: finished saving model to mobilenetv2_eye_best.h5
398/398 ━━━━━━━━━━━━━━━━━━━━ 733s 2s/step - accuracy: 0.9691 - loss: 0.0841 - val_accuracy: 0.9708 - val_loss: 0.0789 - learning_rate: 0.0010


In [ ]:
import tensorflow as tf

print(tf.config.list_physical_devices('GPU'))

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [ ]:
print(max(history.history['val_accuracy']))

0.9707891345024109


In [ ]:
loss, acc = model.evaluate(test_generator)
print("Test Accuracy:", acc)

133/133 ━━━━━━━━━━━━━━━━━━━━ 58s 440ms/step - accuracy: 0.9689 - loss: 0.0832
Test Accuracy: 0.9689064025878906


In [ ]:
preds = model.predict(
    test_generator
)

preds = (preds > 0.5).astype(int)

print(
    classification_report(
        test_generator.classes,
        preds
    )
)

133/133 ━━━━━━━━━━━━━━━━━━━━ 57s 318ms/step
              precision    recall  f1-score   support

           0       0.97      0.97      0.97      8591
           1       0.97      0.96      0.97      8390

    accuracy                           0.97     16981
   macro avg       0.97      0.97      0.97     16981
weighted avg       0.97      0.97      0.97     16981



In [ ]:
model.save(
    "mobilenetv2_eye_classifier.keras"
)

print("Model Saved")

Model Saved
